# 58 - Rank-Band Coverage Check and Relevance-Review Export for Kerem & Manav

Two things happen here. Part A checks how our existing gold-verified labels are actually distributed across the ranked candidate list for the two representative queries we're sending out (query 1 = software companies, query 92 = wealth management firms), since that determines whether we can already claim the top of the ranking is trustworthy or not. Part B builds the blind, un-flagged export files (company details + a blank relevance column) covering the full ranked list per query, in rank order, so Kerem and Manav can label top-down as far as they have time for. Rows we've already gold-labeled sit at their natural rank position in these exports, unflagged, so agreement or disagreement on those rows doubles as a free inter-rater check against our own labeling.

In [1]:
import pandas as pd
from pathlib import Path

OUTPUT_DIR = Path("result/58_gold_label_review_export")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

QUERY_IDS = {1: "software companies", 92: "wealth management firms"}

base_gold = pd.read_json("result/40_active_learning_labeling_queue/expanded_gold_labels.json")
round_gold = pd.read_json("result/42_headline_query_deepening/round_gold_labels.json")
gold = pd.concat([base_gold[["query_id", "domain", "gold_label"]], round_gold[["query_id", "domain", "gold_label"]]], ignore_index=True).drop_duplicates(subset=["query_id", "domain"])
gold_lookup = gold.set_index(["query_id", "domain"])["gold_label"]

silver = pd.read_json("result/55_silver_labels_refreshed/silver_labels_top1000.json")
corpus = pd.read_parquet("result/44_build_scaled_corpus/combined_pool.parquet")[["domain", "name", "organization_type", "country", "summary"]]

print(f"Silver-ranked candidates loaded: {len(silver):,} rows across {silver['query_id'].nunique()} queries")
print(f"Combined gold standard: {len(gold):,} candidates")

Silver-ranked candidates loaded: 101,000 rows across 101 queries
Combined gold standard: 2,269 candidates


## Part A -- Where does our existing gold coverage actually sit in the ranking?

For each query, the ranked top-1000 list is already sorted by the classifier's predicted probability (rank 1 = most confident). This checks, band by band, how many of those ranked candidates already carry a verified gold label, and what fraction of the verified ones are actually relevant. If gold coverage clusters low (large rank numbers) rather than high, it means we genuinely don't have ground truth yet at the top of the ranking, which is exactly the gap the external review needs to close.

In [2]:
RANK_BANDS = [(1, 100), (101, 300), (301, 500), (501, 700), (701, 1000)]

coverage_rows = []
for qid, qname in QUERY_IDS.items():
    q = silver[silver["query_id"] == qid].reset_index(drop=True)
    q["rank"] = q.index + 1  # already sorted descending by silver_prob_relevant
    q["gold_label"] = q.set_index(["query_id", "domain"]).index.map(gold_lookup).values
    gq = q[q["gold_label"].notna()]
    for lo, hi in RANK_BANDS:
        band = gq[(gq["rank"] >= lo) & (gq["rank"] <= hi)]
        coverage_rows.append({
            "query_id": qid, "query": qname, "rank_band": f"{lo}-{hi}",
            "n_gold_verified": len(band),
            "pct_highly_relevant": round((band["gold_label"] >= 2).mean() * 100, 1) if len(band) else None,
            "pct_at_least_partial": round((band["gold_label"] >= 1).mean() * 100, 1) if len(band) else None,
        })

coverage_table = pd.DataFrame(coverage_rows)
coverage_table.to_csv(OUTPUT_DIR / "rank_band_coverage.csv", index=False)
print(coverage_table.to_string(index=False))

 query_id                   query rank_band  n_gold_verified  pct_highly_relevant  pct_at_least_partial
        1      software companies     1-100                9                100.0                 100.0
        1      software companies   101-300               12                100.0                 100.0
        1      software companies   301-500                9                100.0                 100.0
        1      software companies   501-700               11                100.0                 100.0
        1      software companies  701-1000               58                 55.2                 100.0
       92 wealth management firms     1-100                0                  NaN                   NaN
       92 wealth management firms   101-300                0                  NaN                   NaN
       92 wealth management firms   301-500                0                  NaN                   NaN
       92 wealth management firms   501-700                0    

## Part B -- Blind, un-flagged export for manual review

Full ranked top-1000 per query, in rank order, with company details attached and a blank relevance column for Kerem/Manav to fill in. No column reveals our own predicted probability, silver/gold tier, or existing gold label -- rows we've already gold-labeled sit at their natural rank position, indistinguishable from the rest, so their judgments on those rows are a genuinely blind check of our own labeling quality. Rank order alone already front-loads exactly the part of the list we have the least verification for (see Part A), so no extra sampling logic is needed; reviewers can simply work top-down for as long as they have time.

In [3]:
REVIEW_COLUMNS = ["rank", "domain", "name", "organization_type", "country", "summary", "relevance_label"]

for qid, qname in QUERY_IDS.items():
    q = silver[silver["query_id"] == qid].reset_index(drop=True)
    q["rank"] = q.index + 1
    export = q[["rank", "domain"]].merge(corpus, on="domain", how="left")
    export["relevance_label"] = ""  # blank for the reviewer to fill in, see the legend sheet for the 0/1/2 scale
    export = export[REVIEW_COLUMNS]

    safe_name = qname.replace(" ", "_")
    out_path = OUTPUT_DIR / f"review_export_q{qid}_{safe_name}.xlsx"
    with pd.ExcelWriter(out_path, engine="openpyxl") as writer:
        export.to_excel(writer, sheet_name="review", index=False)
        legend = pd.DataFrame({"relevance_label": [0, 1, 2], "meaning": ["not relevant", "partially relevant", "highly relevant"]})
        legend.to_excel(writer, sheet_name="legend", index=False)
    print(f"Saved -> {out_path}  ({len(export)} rows)")

Saved -> result/58_gold_label_review_export/review_export_q1_software_companies.xlsx  (1000 rows)
Saved -> result/58_gold_label_review_export/review_export_q92_wealth_management_firms.xlsx  (1000 rows)


## Summary for the email to Kerem and Manav

In [4]:
print("Export summary")
print("=" * 60)
for qid, qname in QUERY_IDS.items():
    q = silver[silver["query_id"] == qid]
    n_gold_already = q.set_index(["query_id", "domain"]).index.map(gold_lookup).notna().sum()
    print(f"Query {qid} ({qname}): 1000 candidates, rank-ordered, {n_gold_already} already have an internal gold label (not flagged in the export)")
print()
print("Suggested framing for Kerem/Manav: label top-down for as long as you have time for --")
print("even partial coverage (e.g. the first 200-300 per query) is valuable, since that's exactly")
print("the part of the ranking we don't have independent verification for yet.")

Export summary
Query 1 (software companies): 1000 candidates, rank-ordered, 99 already have an internal gold label (not flagged in the export)
Query 92 (wealth management firms): 1000 candidates, rank-ordered, 70 already have an internal gold label (not flagged in the export)

Suggested framing for Kerem/Manav: label top-down for as long as you have time for --
even partial coverage (e.g. the first 200-300 per query) is valuable, since that's exactly
the part of the ranking we don't have independent verification for yet.


## Part C -- A smaller, curated 100-company export prioritizing informativeness

The full 1000-row files above are still the primary export. This adds a separate, much shorter 100-row-per-query alternative for a lighter ask, curated rather than a plain top-100 (which would only ever show the reviewer the ranking's best face and prove nothing about whether it degrades gracefully). 20 candidates are drawn from each of the same five rank bands used in Part A, so the full rank range is still represented, and within each band candidates **not already present in GOI's own production results for that query** are prioritized first, falling back to GOI-overlapping candidates only if a band runs short. That targets exactly the open question from the honest-synthesis discussion: does this pipeline surface genuinely relevant companies production misses entirely, or just noise? Already-gold-labelled rows are still included wherever they naturally fall, unflagged, preserving the same blind consistency check as Part B.

In [5]:
PER_BAND_QUOTA = 20

production = pd.read_excel("dataset/production_results.xlsx")[["query_id", "domain"]]
production_domains = {qid: set(production[production["query_id"] == qid]["domain"]) for qid in QUERY_IDS}

for qid, qname in QUERY_IDS.items():
    q = silver[silver["query_id"] == qid].reset_index(drop=True)
    q["rank"] = q.index + 1
    in_goi = production_domains[qid]

    selected_parts = []
    for lo, hi in RANK_BANDS:
        band = q[(q["rank"] >= lo) & (q["rank"] <= hi)].copy()
        band["in_goi"] = band["domain"].isin(in_goi)
        # Not-in-GOI candidates first (most informative -- tests the coverage-gain claim directly),
        # GOI-overlapping candidates only fill remaining slots, both groups kept in rank order.
        band = pd.concat([band[~band["in_goi"]], band[band["in_goi"]]], ignore_index=True)
        selected_parts.append(band.head(PER_BAND_QUOTA))

    selected = pd.concat(selected_parts, ignore_index=True).sort_values("rank")
    export = selected[["rank", "domain"]].merge(corpus, on="domain", how="left")
    export["relevance_label"] = ""
    export = export[REVIEW_COLUMNS]

    safe_name = qname.replace(" ", "_")
    out_path = OUTPUT_DIR / f"review_export_best100_q{qid}_{safe_name}.xlsx"
    with pd.ExcelWriter(out_path, engine="openpyxl") as writer:
        export.to_excel(writer, sheet_name="review", index=False)
        legend = pd.DataFrame({"relevance_label": [0, 1, 2], "meaning": ["not relevant", "partially relevant", "highly relevant"]})
        legend.to_excel(writer, sheet_name="legend", index=False)

    n_not_in_goi = (~selected["domain"].isin(in_goi)).sum()
    n_gold_already = selected.set_index(["query_id", "domain"]).index.map(gold_lookup).notna().sum()
    print(f"Saved -> {out_path}  ({len(export)} rows)")
    print(f"  not already in GOI production for this query: {n_not_in_goi}/{len(export)}")
    print(f"  already have an internal gold label (not flagged): {n_gold_already}/{len(export)}")

Saved -> result/58_gold_label_review_export/review_export_best100_q1_software_companies.xlsx  (100 rows)
  not already in GOI production for this query: 100/100
  already have an internal gold label (not flagged): 2/100
Saved -> result/58_gold_label_review_export/review_export_best100_q92_wealth_management_firms.xlsx  (100 rows)
  not already in GOI production for this query: 89/100
  already have an internal gold label (not flagged): 0/100
